# Experiment: Delay Time Distribution

Objective:
- Load the delay-time distribution stored in `delta_days_distribution.npz`.
- Summarize the BNS, NSBH, and combined delay-time arrays.
- Produce a publication-ready delay-time distribution figure and save it under `gw-kn-multimodal/figures/`.


In [1]:
import os
_BASE = os.environ.get('BASE_DIR', '/fred/oz016/bgao_kn')

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

BASE = Path(f"{_BASE}/gw-kn-multimodal")
NPZ_PATH = Path(f"{_BASE}/data/Optical_Only_dataset/delta_days_distribution.npz")
OUTDIR = BASE / "figures" / "delay_time_distribution"
OUTDIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.size": 16,
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 16,
    "figure.dpi": 300,
})

NPZ_PATH, OUTDIR

(PosixPath('/fred/oz016/bgao_kn/data/Optical_Only_dataset/delta_days_distribution.npz'),
 PosixPath('/fred/oz016/bgao_kn/gw-kn-multimodal/figures/delay_time_distribution'))

## Plan

- Read the three delay-time arrays from the NPZ file.
- Compute a compact statistical summary.
- Plot normalized histograms for BNS, NSBH, and the combined sample on the same axes.
- Mark `\Delta t = 0` to distinguish negative and positive delays.


In [2]:
data = np.load(NPZ_PATH)
bns = np.asarray(data["delta_days_bns"], dtype=np.float64)
nsbh = np.asarray(data["delta_days_nsbh"], dtype=np.float64)
combined = np.asarray(data["delta_days_combined"], dtype=np.float64)

snr_threshold = float(np.asarray(data["snr_detection_threshold"]))
merge_window_hours = float(np.asarray(data["merge_window_hours"]))

summary = pd.DataFrame([
    {
        "population": "BNS",
        "count": bns.size,
        "mean_days": np.mean(bns),
        "median_days": np.median(bns),
        "p05_days": np.quantile(bns, 0.05),
        "p95_days": np.quantile(bns, 0.95),
    },
    {
        "population": "NSBH",
        "count": nsbh.size,
        "mean_days": np.mean(nsbh),
        "median_days": np.median(nsbh),
        "p05_days": np.quantile(nsbh, 0.05),
        "p95_days": np.quantile(nsbh, 0.95),
    },
    {
        "population": "Combined",
        "count": combined.size,
        "mean_days": np.mean(combined),
        "median_days": np.median(combined),
        "p05_days": np.quantile(combined, 0.05),
        "p95_days": np.quantile(combined, 0.95),
    },
])
summary

,population,count,mean_days,median_days,p05_days,p95_days
0,BNS,16496052,1.516268,1.195094,0.282557,3.783132
1,NSBH,5782736,1.826084,1.549300,0.527900,4.073400
2,Combined,22278788,1.596685,1.294150,0.323000,3.885900


In [3]:
def plot_delay_distribution(bns: np.ndarray, nsbh: np.ndarray, combined: np.ndarray, outpath: Path) -> Path:
    q_lo = float(np.quantile(combined, 0.001))
    q_hi = float(np.quantile(combined, 0.999))
    bins = np.linspace(q_lo, q_hi, 90)

    fig, ax = plt.subplots(figsize=(9.2, 5.8))
    ax.hist(
        combined,
        bins=bins,
        density=True,
        histtype="stepfilled",
        alpha=0.18,
        color="#374151",
        edgecolor="#374151",
        linewidth=1.0,
        label="Combined",
    )
    ax.hist(
        bns,
        bins=bins,
        density=True,
        histtype="step",
        color="#2563EB",
        linewidth=2.0,
        label="BNS",
    )
    ax.hist(
        nsbh,
        bins=bins,
        density=True,
        histtype="step",
        color="#DC2626",
        linewidth=2.0,
        label="NSBH",
    )

    ax.axvline(0.0, color="#111827", linewidth=1.2, linestyle="--", alpha=0.8)
    ax.set_xlabel(r"$\Delta t\,[\mathrm{days}]$")
    ax.set_ylabel("Probability density")
    ax.set_title("Delay-time distribution")
    ax.legend(frameon=False, ncol=3, loc="upper right")

    fig.tight_layout()
    fig.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close(fig)
    return outpath

## Generate Figure


In [4]:
png_path = OUTDIR / "delay_time_distribution.png"
pdf_path = OUTDIR / "delay_time_distribution.pdf"
svg_path = OUTDIR / "delay_time_distribution.svg"

plot_delay_distribution(bns, nsbh, combined, png_path)
plot_delay_distribution(bns, nsbh, combined, pdf_path)
plot_delay_distribution(bns, nsbh, combined, svg_path)

{
    "png": str(png_path),
    "pdf": str(pdf_path),
    "svg": str(svg_path),
}

{'png': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/delay_time_distribution/delay_time_distribution.png',
 'pdf': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/delay_time_distribution/delay_time_distribution.pdf',
 'svg': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/delay_time_distribution/delay_time_distribution.svg'}

In [5]:
result = {
    "npz_file": str(NPZ_PATH),
    "snr_threshold": snr_threshold,
    "merge_window_hours": merge_window_hours,
    "png": str(png_path),
    "pdf": str(pdf_path),
    "svg": str(svg_path),
}
result

{'npz_file': '/fred/oz016/bgao_kn/data/Optical_Only_dataset/delta_days_distribution.npz',
 'snr_threshold': 5.0,
 'merge_window_hours': 2.0,
 'png': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/delay_time_distribution/delay_time_distribution.png',
 'pdf': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/delay_time_distribution/delay_time_distribution.pdf',
 'svg': '/fred/oz016/bgao_kn/gw-kn-multimodal/figures/delay_time_distribution/delay_time_distribution.svg'}

## Results

- The notebook reads `delta_days_bns`, `delta_days_nsbh`, and `delta_days_combined` directly from the NPZ file.
- The output figure overlays BNS and NSBH distributions with the combined distribution as a shaded reference.
- Both PNG and SVG versions are saved under `gw-kn-multimodal/figures/delay_time_distribution/`.
